# Figure 6 New New TrpB D-F Builder

Run this notebook on a separate worker to generate and process the long-running D-F products for TrpB. The final `figure_6_new_new.ipynb` requires the processed pickles written here.


## Setup


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import subprocess
import sys
from pathlib import Path

import jax.random as jr
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import PercentFormatter

from slide.data_generation import (
    nk_grid_pairs,
    ordered_unique_pairs,
    random_start,
    run_nk_start_averaged_diffusion,
)
from slide.direvo_functions import CODON_MAPPER, get_single_decay_rate, get_single_decay_rate_IK_v2
from slide.ruggedness_functions import get_dirichlet_metric, get_nk_l_o_shape
from slide.utils import (
    FIGURE_LABEL_SIZE,
    FIGURE_LEGEND_SIZE,
    FIGURE_TICK_SIZE,
    FIGURE_TITLE_SIZE,
    PANEL_LETTER_SIZE,
    get_figures_dir,
    get_processed_data_dir,
    get_raw_data_dir,
    load_pickle,
    save_pickle,
)
from slide_config import get_slide_data_dir

OVERWRITE_RAW_PKL: bool = False
OVERWRITE_PROCESSED_PKL: bool = False
PLOT_ONLY: bool = False
SAVE_FIGURES: bool = True
PANEL_DPI: int = 350
SAVE_TYPES: tuple[str, ...] = ("pdf", "png", "eps")

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
SLIDE_DATA_DIR = Path(get_slide_data_dir())
REPO_ROOT = Path.cwd()

MODEL_KEYS: tuple[str, ...] = (
    "nuc_uniform",
    "nuc_e_coli_weighted",
    "nuc_e_coli_directed",
)
MODEL_TITLES = {
    "nuc_uniform": "Uniform mutation",
    "nuc_e_coli_weighted": r"Weighted $\mathit{E.\ coli}$",
    "nuc_e_coli_directed": r"Directed $\mathit{E.\ coli}$",
    "nuc_a_thaliana_weighted": r"Weighted $\mathit{A.\ thaliana}$",
    "nuc_a_thaliana_directed": r"Directed $\mathit{A.\ thaliana}$",
}
DF_ECOLI_MODEL_KEYS: tuple[str, ...] = MODEL_KEYS
DF_ATHALIANA_MODEL_KEYS: tuple[str, ...] = (
    "nuc_uniform",
    "nuc_a_thaliana_weighted",
    "nuc_a_thaliana_directed",
)
DF_ALL_MODEL_KEYS: tuple[str, ...] = tuple(dict.fromkeys(DF_ECOLI_MODEL_KEYS + DF_ATHALIANA_MODEL_KEYS))
LANDSCAPE_KEYS: tuple[str, ...] = ("trpb",)
LANDSCAPE_NAMES: tuple[str, ...] = ("TrpB",)
LANDSCAPE_COLORS: tuple[str, ...] = ("tab:blue",)
LANDSCAPE_MARKERS: tuple[str, ...] = ("s",)

ABC_PROCESSED_PATH = PROCESSED_DATA_DIR / "figure6_ecoli_nk_local_gmu_processed.pkl"
DF_ECOLI_PER_LANDSCAPE_PROCESSED_PATHS = {
    landscape: PROCESSED_DATA_DIR / f"figure6_full_nuc_gmu_ecoli_{landscape}_sampling_75steps_processed.pkl"
    for landscape in LANDSCAPE_KEYS
}
DF_ATHALIANA_PER_LANDSCAPE_PROCESSED_PATHS = {
    landscape: PROCESSED_DATA_DIR / f"figure6_full_nuc_gmu_a_thaliana_{landscape}_sampling_75steps_processed.pkl"
    for landscape in LANDSCAPE_KEYS
}
ANALYTICS_PER_LANDSCAPE_PATHS = {
    landscape: PROCESSED_DATA_DIR / f"figure6_full_nuc_gmu_{landscape}_kernel_analytics_processed.pkl"
    for landscape in LANDSCAPE_KEYS
}
LANDSCAPE_FILE_BY_NAME = {
    "TrpB": "TrpB_landscape_array.pkl",
}
ABC_RAW_PATHS = {
    model: RAW_DATA_DIR / f"figure6_ecoli_nk_{model}_raw.pkl"
    for model in MODEL_KEYS
}

# Panels A-C: retained Figure S2 design.
ABC_N_VALUES: tuple[int, ...] = (10, 14, 18, 23, 27, 32, 36, 41, 45, 50)
ABC_NUM_ALLELES: int = 4
ABC_NUM_K_VALUES_PER_N: int = 10
ABC_NUM_LANDSCAPES: int = 25
ABC_NUM_STARTS: int = 25
ABC_NUM_REPLICATES: int = 5
ABC_POPULATION_SIZE: int = 2_500
ABC_NUM_GENERATIONS: int = 25
ABC_TOTAL_MUTATION_RATE: float = 0.5

# Panels D-F and S3.
DF_NUM_GENERATIONS: int = 75
DF_TOTAL_MUTATION_RATE: float = 0.1
RANDOM_SEED: int = 42

# Existing panels G-H.
RAW_PATH = RAW_DATA_DIR / "figure6_new_nk_landscapes.pkl"
PROCESSED_PATH = PROCESSED_DATA_DIR / "figure6_new_lethal_neutral_rho2.pkl"
DECAY_RAW_PATH = RAW_DATA_DIR / "figure6_new_new_lethal_neutral_gmu_global_local_raw.pkl"
DECAY_PROCESSED_PATH = PROCESSED_DATA_DIR / "figure6_new_new_lethal_neutral_gmu_global_local_rho2.pkl"
N_SITES: int = 4
NUM_ALLELES: int = 20
K_VALUES: tuple[int, ...] = (0, 1, 2, 3)
FRACTIONS: tuple[float, ...] = (0.0, 0.05, 0.10, 0.20, 0.35, 0.50)
NUM_LANDSCAPES: int = 20
NUM_LOCAL_STARTS: int = 20
TOTAL_MUTATION_RATE: float = 0.1
NUM_DECAY_STEPS: int = 75
PERTURBATIONS: tuple[str, ...] = ("lethal", "neutral")

print(f"PLOT_ONLY={PLOT_ONLY}, OVERWRITE_RAW_PKL={OVERWRITE_RAW_PKL}, "
      f"OVERWRITE_PROCESSED_PKL={OVERWRITE_PROCESSED_PKL}")


def save_figure(fig: Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save one figure in every configured format.

    Parameters:
    - fig: Figure
        Figure to save.
    - stem: str
        Output filename stem.
    - bbox_inches: str
        Matplotlib bounding-box mode.

    Returns:
    - None
        Files are written below ``FIGURES_DIR``.
    """
    for suffix in SAVE_TYPES:
        destination = FIGURES_DIR / suffix
        destination.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination / f"{stem}.{suffix}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


def add_panel_letter(ax: Axes, letter: str) -> None:
    """Add a manuscript panel letter.

    Parameters:
    - ax: Axes
        Axis receiving the label.
    - letter: str
        Panel letter.

    Returns:
    - None
        The axis is modified in place.
    """
    ax.text(-0.14, 1.10, letter, transform=ax.transAxes,
            fontsize=PANEL_LETTER_SIZE, fontweight="bold", va="top", ha="left")


## Mutation Kernels


In [ ]:
def symmetric_sinkhorn_kernel(kernel: np.ndarray, tolerance: float = 1e-13) -> np.ndarray:
    """Create a symmetric doubly-stochastic kernel by diagonal scaling.

    Parameters:
    - kernel: np.ndarray
        Non-negative square base kernel.
    - tolerance: float
        Maximum permitted row-sum error.

    Returns:
    - np.ndarray
        Symmetric, doubly-stochastic kernel with the input zero pattern.
    """
    symmetric = 0.5 * (np.asarray(kernel, dtype=float) + np.asarray(kernel, dtype=float).T)
    scale = np.ones(symmetric.shape[0], dtype=float)
    for _ in range(100_000):
        row_sums = scale * (symmetric @ scale)
        if np.max(np.abs(row_sums - 1.0)) < tolerance:
            break
        scale *= np.sqrt(1.0 / row_sums)
    else:
        raise RuntimeError("Symmetric Sinkhorn scaling did not converge.")
    result = scale[:, None] * symmetric * scale[None, :]
    return result


def stationary_distribution(kernel: np.ndarray) -> np.ndarray:
    """Return the normalized stationary distribution of a row-stochastic kernel.

    Parameters:
    - kernel: np.ndarray
        Irreducible row-stochastic transition kernel.

    Returns:
    - np.ndarray
        Positive stationary probability vector.
    """
    eigenvalues, eigenvectors = np.linalg.eig(np.asarray(kernel, dtype=float).T)
    index = int(np.argmin(np.abs(eigenvalues - 1.0)))
    stationary = np.real(eigenvectors[:, index])
    if stationary.sum() < 0:
        stationary *= -1
    stationary /= stationary.sum()
    return stationary


uniform_kernel = (np.ones((4, 4)) - np.eye(4)) / 3.0
e_coli_directed_kernel = np.asarray(
    np.load(REPO_ROOT / "other_data" / "normed_e_coli_matrix.npy"), dtype=float
)
e_coli_weighted_kernel = symmetric_sinkhorn_kernel(e_coli_directed_kernel)
a_thaliana_directed_kernel = np.asarray(
    np.load(REPO_ROOT / "other_data" / "normed_a_thaliana_matrix.npy"), dtype=float
)
a_thaliana_weighted_kernel = symmetric_sinkhorn_kernel(a_thaliana_directed_kernel)
MUTATION_KERNELS = {
    "nuc_uniform": uniform_kernel,
    "nuc_e_coli_weighted": e_coli_weighted_kernel,
    "nuc_e_coli_directed": e_coli_directed_kernel,
}

for model, kernel in MUTATION_KERNELS.items():
    if kernel.shape != (4, 4) or np.any(kernel < 0):
        raise ValueError(f"Invalid mutation kernel for {model}.")
    if not np.allclose(kernel.sum(axis=1), 1.0, atol=1e-12):
        raise ValueError(f"Mutation kernel {model} is not row-stochastic.")
    adjacency = kernel > 0
    reachability = np.eye(4, dtype=bool)
    power = np.eye(4, dtype=bool)
    for _ in range(1, 4):
        power = (power.astype(int) @ adjacency.astype(int)) > 0
        reachability |= power
    if not np.all(reachability):
        raise ValueError(f"Mutation kernel {model} is not irreducible.")
if not np.allclose(e_coli_weighted_kernel, e_coli_weighted_kernel.T, atol=1e-12):
    raise ValueError("Weighted E. coli kernel is not symmetric.")
if not np.allclose(e_coli_weighted_kernel.sum(axis=0), 1.0, atol=1e-12):
    raise ValueError("Weighted E. coli kernel is not column-stochastic.")
if not np.allclose(np.diag(e_coli_weighted_kernel), 0.0, atol=1e-14):
    raise ValueError("Weighted E. coli kernel does not preserve the zero diagonal.")

ALL_DF_MUTATION_KERNELS = {
    **MUTATION_KERNELS,
    "nuc_a_thaliana_weighted": a_thaliana_weighted_kernel,
    "nuc_a_thaliana_directed": a_thaliana_directed_kernel,
}

for model, kernel in ALL_DF_MUTATION_KERNELS.items():
    if kernel.shape != (4, 4) or np.any(kernel < 0):
        raise ValueError(f"Invalid mutation kernel for {model}.")
    if not np.allclose(kernel.sum(axis=1), 1.0, atol=1e-12):
        raise ValueError(f"Mutation kernel {model} is not row-stochastic.")
if not np.allclose(a_thaliana_weighted_kernel, a_thaliana_weighted_kernel.T, atol=1e-12):
    raise ValueError("Weighted A. thaliana kernel is not symmetric.")
if not np.allclose(a_thaliana_weighted_kernel.sum(axis=0), 1.0, atol=1e-12):
    raise ValueError("Weighted A. thaliana kernel is not column-stochastic.")
if not np.allclose(np.diag(a_thaliana_weighted_kernel), 0.0, atol=1e-14):
    raise ValueError("Weighted A. thaliana kernel does not preserve the zero diagonal.")

STATIONARY_DISTRIBUTIONS = {
    model: stationary_distribution(kernel)
    for model, kernel in MUTATION_KERNELS.items()
}


## Raw Full-Nucleotide Products


In [ ]:
FULL_NUC_RAW_PATHS = {
    model: [
        RAW_DATA_DIR / f"figure6_full_nuc_gmu_{landscape}_{model}_raw.pkl"
        for landscape in LANDSCAPE_KEYS
    ]
    for model in DF_ALL_MODEL_KEYS
}


def get_missing_full_nuc_raw_paths(models: tuple[str, ...]) -> list[Path]:
    """Return missing full-nucleotide G_mu payload paths for a model group.

    Parameters:
    - models: tuple[str, ...]
        Mutation models to inspect.

    Returns:
    - list[Path]
        Missing raw-derived payload paths.
    """
    return [path for model in models for path in FULL_NUC_RAW_PATHS[model] if not path.exists()]


def generate_missing_full_nuc_raw_products(models: tuple[str, ...]) -> None:
    """Generate missing full-nucleotide G_mu payloads for this landscape.

    Parameters:
    - models: tuple[str, ...]
        Mutation models to pass to the raw generator.

    Returns:
    - None
        Writes raw-derived payloads when generation is requested.
    """
    command = [
        sys.executable,
        str(Path("scripts/generate_figure6_full_nuc_gmu.py")),
        "--landscapes",
        ",".join(LANDSCAPE_KEYS),
        "--models",
        ",".join(models),
        "--output-dir",
        str(RAW_DATA_DIR),
    ]
    if OVERWRITE_RAW_PKL:
        command.append("--overwrite")
    subprocess.run(command, check=True)


def report_full_nuc_raw_products(models: tuple[str, ...], label: str) -> list[Path]:
    """Print and return missing full-nucleotide products for a model group.

    Parameters:
    - models: tuple[str, ...]
        Mutation models required by a D-F panel group.
    - label: str
        Human-readable group label used in messages.

    Returns:
    - list[Path]
        Missing raw-derived payload paths after any requested generation.
    """
    missing_paths = get_missing_full_nuc_raw_paths(models)
    if not PLOT_ONLY and (missing_paths or OVERWRITE_RAW_PKL):
        generate_missing_full_nuc_raw_products(models)
        missing_paths = get_missing_full_nuc_raw_paths(models)
    print(f"Missing {label} full-nucleotide D-F raw products: {len(missing_paths)}")
    for path in missing_paths:
        print(f"  {path}")
    return missing_paths


def ensure_full_nuc_raw_products(models: tuple[str, ...], label: str) -> None:
    """Raise if raw full-nucleotide products are unavailable for processing.

    Parameters:
    - models: tuple[str, ...]
        Mutation models required by a D-F panel group.
    - label: str
        Human-readable group label used in error messages.

    Returns:
    - None
        Raises when products are missing.
    """
    missing_paths = get_missing_full_nuc_raw_paths(models)
    if missing_paths:
        listing = "\n".join(str(path) for path in missing_paths)
        raise FileNotFoundError(f"Missing {label} full-nucleotide D-F raw products:\n{listing}")


missing_ecoli_full_nuc_raw_paths = report_full_nuc_raw_products(DF_ECOLI_MODEL_KEYS, "E. coli")
missing_a_thaliana_full_nuc_raw_paths = report_full_nuc_raw_products(DF_ATHALIANA_MODEL_KEYS, "A. thaliana")


## Analytical Rates And Asymptotes


In [ ]:
def build_nucleotide_landscape(landscape: np.ndarray) -> np.ndarray:
    """Expand an amino-acid landscape into nucleotide/codon space.

    Parameters:
    - landscape: np.ndarray
        Amino-acid landscape with one axis per residue.

    Returns:
    - np.ndarray
        Nucleotide landscape with three four-state axes per residue.
    """
    mapper = np.asarray(CODON_MAPPER, dtype=np.int16)
    buffered = np.pad(
        np.asarray(landscape, dtype=np.float64),
        [(0, 1)] * landscape.ndim,
        constant_values=float(np.min(landscape)),
    )
    indices = np.indices((4,) * (3 * landscape.ndim), dtype=np.int8)
    amino_acids = [
        mapper[indices[3 * site], indices[3 * site + 1], indices[3 * site + 2]]
        for site in range(landscape.ndim)
    ]
    return buffered[tuple(amino_acids)]


def apply_kernel_axis(values: np.ndarray, kernel: np.ndarray, axis: int) -> np.ndarray:
    """Apply one row-stochastic kernel to one function axis.

    Parameters:
    - values: np.ndarray
        Genotype-indexed function values.
    - kernel: np.ndarray
        Single-site row-stochastic kernel.
    - axis: int
        Axis receiving the kernel.

    Returns:
    - np.ndarray
        Kernel-transformed function values.
    """
    transformed = np.tensordot(kernel, values, axes=([1], [axis]))
    return np.moveaxis(transformed, 0, axis)


def stationary_average(values: np.ndarray, stationary: np.ndarray) -> float:
    """Compute the product-stationary average without forming a tensor product.

    Parameters:
    - values: np.ndarray
        Nucleotide-space landscape.
    - stationary: np.ndarray
        Single-site stationary distribution.

    Returns:
    - float
        Product-distribution weighted average.
    """
    contracted = np.asarray(values, dtype=np.float64)
    for _ in range(values.ndim):
        contracted = np.tensordot(stationary, contracted, axes=([0], [0]))
    return float(contracted)


def analytical_squared_decay(values: np.ndarray, kernel: np.ndarray, directed: bool) -> dict[str, object]:
    """Compute the analytical squared-decay metric for a product kernel.

    Parameters:
    - values: np.ndarray
        Nucleotide-space fitness landscape.
    - kernel: np.ndarray
        Single-site transition kernel.
    - directed: bool
        Whether to use the symmetrized directed Laplacian and stationary asymptote.

    Returns:
    - dict[str, object]
        Rate, asymptote, power term, and stationary distribution.
    """
    stationary = stationary_distribution(kernel)
    mean = stationary_average(values, stationary) if directed else float(values.mean())
    b_zero = float(values.size * mean * mean)
    denominator = float(np.vdot(values, values).real - b_zero)
    if denominator <= 0:
        raise ValueError("Squared-decay denominator must be positive.")
    laplacian_values = np.zeros_like(values, dtype=np.float64)
    for axis in range(values.ndim):
        forward = values - apply_kernel_axis(values, kernel, axis)
        if directed:
            reverse = values - apply_kernel_axis(values, kernel.T, axis)
            laplacian_values += 0.5 * (forward + reverse)
        else:
            laplacian_values += forward
    numerator = float(np.vdot(values, laplacian_values).real)
    rho_2 = numerator / (values.ndim * denominator)
    return {
        "rho_2": rho_2,
        "G_infinity": mean * mean,
        "b_0": b_zero,
        "stationary_distribution": stationary,
        "numerator_reduced": numerator,
    }


def analytics_complete(payload: dict[str, object]) -> bool:
    """Return whether an analytics payload covers this landscape and all D-F models.

    Parameters:
    - payload: dict[str, object]
        Candidate analytics payload.

    Returns:
    - bool
        Whether every configured model entry is present.
    """
    data = payload.get("data", {})
    return all(
        name in data and all(model in data[name] for model in DF_ALL_MODEL_KEYS)
        for name in LANDSCAPE_NAMES
    )


analytics_path = ANALYTICS_PER_LANDSCAPE_PATHS[LANDSCAPE_KEYS[0]]
if analytics_path.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure6_analytics = load_pickle(analytics_path)
    if not analytics_complete(figure6_analytics):
        if PLOT_ONLY:
            raise FileNotFoundError(f"PLOT_ONLY=True requires complete analytical payload: {analytics_path}")
        figure6_analytics = None
elif PLOT_ONLY:
    raise FileNotFoundError(f"PLOT_ONLY=True requires {analytics_path}")
else:
    figure6_analytics = None

if figure6_analytics is None:
    analytical_data: dict[str, object] = {}
    for landscape_name in LANDSCAPE_NAMES:
        filename = LANDSCAPE_FILE_BY_NAME[landscape_name]
        with (REPO_ROOT / "landscape_arrays" / filename).open("rb") as handle:
            amino_acid_landscape = np.asarray(pickle.load(handle))
        nucleotide_landscape = build_nucleotide_landscape(amino_acid_landscape)
        analytical_data[landscape_name] = {
            model: analytical_squared_decay(
                nucleotide_landscape,
                ALL_DF_MUTATION_KERNELS[model],
                directed=model.endswith("_directed"),
            )
            for model in DF_ALL_MODEL_KEYS
        }
    figure6_analytics = {
        "data": analytical_data,
        "params": {
            "models": DF_ALL_MODEL_KEYS,
            "kernels": ALL_DF_MUTATION_KERNELS,
            "normalization": "sum(I-T_i) / N_nucleotide",
        },
        "metadata": {
            "paper_reference": "Figure 6D-F and standalone A. thaliana D-F",
            "weighted_kernel": "symmetric Sinkhorn scaling of transpose-averaged nucleotide kernels",
            "directed_asymptote": "product stationary distribution of directed nucleotide kernels",
        },
    }
    save_pickle(figure6_analytics, analytics_path)


## Processed Sampling Accuracy


In [ ]:
def normalise_curve(curve: np.ndarray) -> np.ndarray:
    """Normalise a G_mu curve by its first generation value.

    Parameters:
    - curve: np.ndarray
        One-dimensional G_mu curve.

    Returns:
    - np.ndarray
        Normalised curve with finite values.
    """
    curve = np.asarray(curve, dtype=float)
    denominator = max(float(curve[0]), 1e-10)
    return curve / denominator


def fit_full_nuc_gmu_curve(curve: np.ndarray) -> float:
    """Fit one full-nucleotide G_mu curve and return rho_2.

    Parameters:
    - curve: np.ndarray
        Unnormalised G_mu curve.

    Returns:
    - float
        Fitted rho_2 value.
    """
    normalised = normalise_curve(curve)
    rho_raw = get_single_decay_rate_IK_v2(
        normalised,
        mut=DF_TOTAL_MUTATION_RATE,
        num_steps=DF_NUM_GENERATIONS,
    )[0]
    return float(rho_raw / 2.0)


def load_full_nuc_raw_by_model(models: tuple[str, ...]) -> dict[str, list[dict[str, object]]]:
    """Load compact full-nucleotide G_mu payloads for a model group.

    Parameters:
    - models: tuple[str, ...]
        Mutation models to load.

    Returns:
    - dict[str, list[dict[str, object]]]
        Raw payloads keyed by model, ordered by ``LANDSCAPE_KEYS``.
    """
    raw_by_model: dict[str, list[dict[str, object]]] = {}
    for model in models:
        raw_by_model[model] = [load_pickle(path) for path in FULL_NUC_RAW_PATHS[model]]
    return raw_by_model


def process_full_nuc_df_payload(
    raw_by_model: dict[str, list[dict[str, object]]],
    models: tuple[str, ...],
) -> dict[str, object]:
    """Fit full-nucleotide squared-decay rates for this landscape.

    Parameters:
    - raw_by_model: dict[str, list[dict[str, object]]]
        Compact full-nucleotide G_mu payloads keyed by mutation model.
    - models: tuple[str, ...]
        Models to process and preserve in output order.

    Returns:
    - dict[str, object]
        Per-model fitted-rate distributions and counts.
    """
    processed: dict[str, object] = {}
    counts_by_model: dict[str, list[np.ndarray]] = {}
    num_fit_failures = 0
    for model in models:
        model_results = []
        model_counts = []
        for landscape, payload in zip(LANDSCAPE_KEYS, raw_by_model[model], strict=True):
            data = payload["data"]
            g_mu = np.asarray(data["g_mu"], dtype=float)
            counts = np.asarray(data["counts"], dtype=int)
            included_counts = np.asarray(data["included_counts"], dtype=int)
            if g_mu.shape != (20, len(counts), DF_NUM_GENERATIONS):
                raise ValueError(f"Unexpected g_mu shape for {landscape}/{model}: {g_mu.shape}")
            if not np.array_equal(included_counts, np.broadcast_to(counts[None, :], included_counts.shape)):
                raise ValueError(f"Included counts do not match counts for {landscape}/{model}.")
            if not np.isfinite(g_mu).all():
                raise ValueError(f"Non-finite g_mu values for {landscape}/{model}.")
            landscape_results = []
            for count_index in range(len(counts)):
                estimates = []
                for ordering_index in range(g_mu.shape[0]):
                    try:
                        estimates.append(fit_full_nuc_gmu_curve(g_mu[ordering_index, count_index]))
                    except RuntimeError:
                        num_fit_failures += 1
                        estimates.append(np.nan)
                landscape_results.append(np.asarray(estimates, dtype=float))
            model_results.append(landscape_results)
            model_counts.append(counts)
        processed[model] = model_results
        counts_by_model[model] = model_counts
    return {
        "data": processed,
        "counts": counts_by_model,
        "params": {
            "models": models,
            "M": DF_NUM_GENERATIONS,
            "total_mutation_rate": DF_TOTAL_MUTATION_RATE,
            "num_orderings": 20,
            "kernels": {model: ALL_DF_MUTATION_KERNELS[model] for model in models},
            "raw_paths": {model: [str(path) for path in FULL_NUC_RAW_PATHS[model]] for model in models},
        },
        "metadata": {
            "paper_reference": "Figure 6D-F",
            "description": "Per-landscape full-nucleotide compact G_mu start-prefix fitted rho_2 distributions.",
            "num_fit_failures": num_fit_failures,
            "standard_deviation_ddof": 1,
        },
    }


def load_or_process_full_nuc_df_payload(
    path: Path,
    models: tuple[str, ...],
    label: str,
) -> dict[str, object]:
    """Load or process one per-landscape full-nucleotide D-F payload.

    Parameters:
    - path: Path
        Processed payload destination.
    - models: tuple[str, ...]
        Mutation models in the payload.
    - label: str
        Human-readable group label.

    Returns:
    - dict[str, object]
        Processed D-F payload.
    """
    if path.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
        return load_pickle(path)
    if PLOT_ONLY:
        raise FileNotFoundError(f"PLOT_ONLY=True requires {path}")
    ensure_full_nuc_raw_products(models, label)
    payload = process_full_nuc_df_payload(load_full_nuc_raw_by_model(models), models)
    save_pickle(payload, path)
    return payload


figure6_df_payload = load_or_process_full_nuc_df_payload(
    DF_ECOLI_PER_LANDSCAPE_PROCESSED_PATHS[LANDSCAPE_KEYS[0]],
    DF_ECOLI_MODEL_KEYS,
    "E. coli",
)
figure6_a_thaliana_df_payload = load_or_process_full_nuc_df_payload(
    DF_ATHALIANA_PER_LANDSCAPE_PROCESSED_PATHS[LANDSCAPE_KEYS[0]],
    DF_ATHALIANA_MODEL_KEYS,
    "A. thaliana",
)
print(f"E. coli fit failures: {figure6_df_payload['metadata']['num_fit_failures']}")
print(f"A. thaliana fit failures: {figure6_a_thaliana_df_payload['metadata']['num_fit_failures']}")
